# Google Colab notebook для ASR-этапа тестового пайплайна (Питер FM)

Этот ноутбук запускает **ASR-only** шаг для тестового фильма **«Питер FM»**.

Отличия от validation-ноутбука:
- нет gold-разметки и окон из Excel;
- фильм разбивается на фиксированные чанки через ;
- ASR считается по каждому  через .

Рекомендуемый режим работы:
- **код** брать из GitHub;
- **данные и артефакты** хранить на Google Drive;
- **ASR** считать в Colab на GPU;
- **diarization / speaker ID / post-processing** считать отдельно.


## 1. Проверка runtime

Перед запуском выберите в Colab:

**Runtime → Change runtime type → T4 GPU**

In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

Sat Mar 28 13:34:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Подключение Google Drive

На Google Drive удобно хранить только **данные** и **артефакты**.

Рекомендуемая структура:

- 
-  — голосовые профили персонажей
-  — папка для артефактов


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 3. Получение кода из GitHub

Рекомендуемый способ:

1. редактировать код локально;
2. делать `git push` в GitHub;
3. в Colab каждый раз делать `git clone` или `git pull`.

Так Colab всегда берёт актуальную версию кода без ручного копирования файлов.

Репозиторий:
`https://github.com/DeMiYak/AudioIntent`

/content/AudioIntent - временная библиотека с кодом для сессии Colab

In [ ]:
from pathlib import Path
import shutil
import subprocess
import os

REPO_URL = "https://github.com/DeMiYak/AudioIntent.git"
REPO_DIR = Path("/content/AudioIntent")
print(REPO_DIR)

if REPO_DIR.exists():
    print("Resetting local changes and pulling latest changes...")
    # Reset local changes to allow git pull
    !git -C "{REPO_DIR}" checkout src/diarization.py
    !git -C "{REPO_DIR}" pull --ff-only
else:
    !git clone "{REPO_URL}" "{REPO_DIR}"

!git -C "{REPO_DIR}" rev-parse --abbrev-ref HEAD
!git -C "{REPO_DIR}" log -1 --oneline

/content/AudioIntent
Cloning into '/content/AudioIntent'...
remote: Enumerating objects: 176, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 176 (delta 92), reused 124 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (176/176), 160.52 KiB | 6.69 MiB/s, done.
Resolving deltas: 100% (92/92), done.
main
3360a79 (HEAD -> main, origin/main, origin/HEAD) Updated char_to_token_spans


## 4. Как корректно загрузить файлы

Для этой задачи лучше разделить **код** и **данные**.

### Что должно лежать на Google Drive

Обязательно:

-  — тестовый фильм
-  — голосовые профили персонажей

### Как загружать

Для больших файлов ():
- положить на Drive с локальной машины заранее;
- не использовать  для больших медиафайлов.

### Что НЕ нужно копировать на Drive

- , ,  — код берётся из GitHub.


In [ ]:
from pathlib import Path
import os

DRIVE_BASE = Path("/content/drive/MyDrive")
DRIVE_DATA_ROOT = DRIVE_BASE / "AudioIntent" / "data"

# Test film paths
TEST_DIR = DRIVE_DATA_ROOT / "raw" / "test"
MEDIA_INPUT = TEST_DIR / "Peter_FM_2006.mkv"
SAMPLES_DIR = TEST_DIR / "audio_profile"

ARTIFACTS_ROOT = DRIVE_DATA_ROOT / "artifacts"
OUTPUT_DIR = ARTIFACTS_ROOT / "test_piter_fm_asr_diarization_colab"
WINDOWS_DIR = OUTPUT_DIR / "windows"

print("--- Path Verification ---")
print(f"Media input : {MEDIA_INPUT} -> {'FOUND' if MEDIA_INPUT.exists() else 'MISSING'}")
print(f"Samples dir : {SAMPLES_DIR} -> {'FOUND' if SAMPLES_DIR.exists() else 'MISSING'}")
print(f"Output dir  : {OUTPUT_DIR}")


Checking contents of: /content/drive/MyDrive/AudioIntent/data
[data/]
    [interim/]
        .gitkeep (1)
        .gitkeep
    [processed/]
        .gitkeep
        rule_lexicon.json
        gold_stats.json
        gold_skipped_lines.json
        gold_dialogues.jsonl
    [raw/]
        [gold/]
            data_val.xlsx

--- Path Verification ---
Looking for Gold XLSX at: /content/drive/MyDrive/AudioIntent/data/raw/gold/data_val.xlsx -> FOUND
Looking for Validation Dir at: /content/drive/MyDrive/AudioIntent/data/raw/validation -> FOUND
Looking for Media Input at: /content/drive/MyDrive/AudioIntent/data/raw/validation/status_svoboden.mkv -> FOUND


## 5. Установка зависимостей для ASR-only прогона

В этом ноутбуке ставится только то, что нужно для `faster-whisper` и служебных шагов проекта.

`pyannote.audio`, diarization и speaker identification здесь специально не поднимаются:
ASR считается отдельно, чтобы снизить риск конфликтов зависимостей в Colab.

In [ ]:
%cd {REPO_DIR}
!apt-get -qq update
!apt-get -qq install -y ffmpeg pkg-config

# Force install jedi first to satisfy ipython requirements
%pip install -q "jedi>=0.16"
%pip install -q -U pip setuptools wheel
%pip install -q --only-binary=:all: "av==15.1.0"
%pip install -q "ctranslate2==4.4.0" "faster-whisper==1.1.1" hf_xet openpyxl pyyaml scikit-learn tqdm

import torch
import av
import ctranslate2
import faster_whisper
import pandas as pd

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("pandas:", pd.__version__)
print("av:", av.__version__)
print("ctranslate2:", ctranslate2.__version__)
print("faster_whisper:", faster_whisper.__version__)

/content/AudioIntent
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 122354 files and directories currently installed.)
Removing r-base-dev (4.5.3-1.2204.0) ...
dpkg: pkgconf: dependency problems, but removing anyway as you requested:
 libsndfile1-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libmkl-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libglib2.0-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config is to be removed.
 libfontconfig-dev:amd64 depends on pkg-config; however:
  Package pkg-config is not installed.
  Package pkgconf which provides pkg-config 

## 6. Разбивка фильма на чанки

Фильм разбивается на чанки по 10 минут. Каждый чанк сохраняется как  в своей папке.
Повторный запуск пропускает уже существующие чанки (используйте  для перезаписи).


In [ ]:
%cd {REPO_DIR}
!python -m src.chunk_film 
  --input "{MEDIA_INPUT}" 
  --output-dir "{WINDOWS_DIR}" 
  --chunk-duration 600

print("Chunks created in:", WINDOWS_DIR)
print("Count:", len(list(WINDOWS_DIR.iterdir())) if WINDOWS_DIR.exists() else 0)


/content/AudioIntent

=== GOLD PREPROCESSING STATS ===
Диалогов: 280
Реплик: 1170
Реплик с разметкой: 588
Всего размеченных выражений: 591
Пропущенных строк: 8

Распределение по типам:
  contact_open: 413
  contact_close: 178

Причины пропусков:
  speaker_header_without_text: 4
  line_has_no_valid_speaker_separator: 4

Топ-20 выражений:
  привет: 37
  здравствуйте: 34
  здрасьте: 24
  да: 19
  до свидания: 15
  пока: 14
  спасибо: 13
  давай: 11
  здорово: 9
  добрый день: 9
  алло: 9
  алё: 7
  здравия желаю: 6
  здрасте: 5
  доброе утро: 5
  олег: 3
  здорова: 3
  кать: 3
  счастливо: 3
  все: 3

Входной Excel: /content/drive/MyDrive/AudioIntent/data/raw/gold/data_val.xlsx
Готово. JSONL сохранён в: /content/drive/MyDrive/AudioIntent/data/processed/gold_dialogues.jsonl
Статистика сохранена в: data/processed/gold_stats.json
Пропущенные строки сохранены в: data/processed/gold_skipped_lines.json
FIT_INPUT exists: True


## 7. Подготовка gold-лексикона для постпроцессинга

Gold-данные (лексикон для rule-based intent) подготавливаются локально и хранятся на Drive.
Этот шаг не нужен в Colab, если  уже лежит на Drive.


In [ ]:
%cd {REPO_DIR}
FIT_INPUT = DRIVE_DATA_ROOT / "processed" / "gold_dialogues.jsonl"
print("gold_dialogues.jsonl exists:", FIT_INPUT.exists())
# If missing, run locally: python -m src.preprocess_gold ...


/content/AudioIntent
Gold для evaluation.ipynb сохранён в: /content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/gold.xlsx
gold.xlsx exists: True


## 8. Hugging Face token

Для скачивания ASR-модели можно использовать секрет Colab.

Рекомендуется:
- открыть вкладку **Secrets** в Colab;
- создать секрет с именем `AudioIntent`;
- не печатать сам токен в output.

In [ ]:
from google.colab import userdata
import os

try:
    os.environ["HF_TOKEN"] = userdata.get("AudioIntent")
    print("HF_TOKEN is set.")
except userdata.SecretNotFoundError:
    print("HF_TOKEN not found in Colab Secrets.")

HF_TOKEN is set.


## 9. Smoke test на одном чанке (ASR-only)

Проверяет, что пути, модель и GPU работают корректно.
Сохраняет  для первого чанка.

Если Colab выдаёт ошибку на , сначала выполните следующую ячейку с установкой .


In [ ]:
!apt-get -qq update
!apt-get -qq install -y --no-install-recommends libcudnn8
print("cuDNN 8 installation attempted. If smoke test fails with cuDNN error, re-run the next cell.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libcudnn8.
(Reading database ... 122346 files and directories currently installed.)
Preparing to unpack .../libcudnn8_8.9.7.29-1+cuda12.2_amd64.deb ...
Unpacking libcudnn8 (8.9.7.29-1+cuda12.2) ...


In [ ]:
%cd {REPO_DIR}
!python -m src.pipeline 
    --scan-windows 
    --transcript-input-dir "{WINDOWS_DIR}" 
    --output-dir "{OUTPUT_DIR}" 
    --only-asr 
    --asr-model-name small 
    --device cuda 
    --compute-type float16 
    --batch-size 4 
    --limit 1


/content/AudioIntent
Validation pipeline завершён. Окон с обработкой: 1
Gold сохранён в: /content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/gold_smoke.xlsx
Режим запуска: asr_only. Итоговый Excel с парами на этом шаге не формировался.


## 10. Основной ASR run по всем чанкам

После успешного smoke test считаем все чанки.
На выходе:  в каждой папке .


In [ ]:
%cd {REPO_DIR}
!python -m src.pipeline 
  --scan-windows 
  --transcript-input-dir "{WINDOWS_DIR}" 
  --output-dir "{OUTPUT_DIR}" 
  --only-asr 
  --asr-model-name medium 
  --device cuda 
  --compute-type float16 
  --batch-size 4


/content/AudioIntent
Validation pipeline завершён. Окон с обработкой: 28
Gold сохранён в: /content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/gold.xlsx
Режим запуска: asr_only. Итоговый Excel с парами на этом шаге не формировался.


## 11. Быстрая проверка артефактов ASR

In [ ]:
import json
from pathlib import Path

windows_dir = OUTPUT_DIR / "windows"
print("windows_dir exists:", windows_dir.exists())

if windows_dir.exists():
    window_dirs = sorted([p for p in windows_dir.iterdir() if p.is_dir()])
    print("window count:", len(window_dirs))

    for p in window_dirs[:5]:
        print("\nWindow:", p.name)
        for name in ["audio.wav", "transcript.json", "summary.json"]:
            print(f" - {name}: {(p / name).exists()}")

    transcripts = [p / "transcript.json" for p in window_dirs if (p / "transcript.json").exists()]
    print("\ntranscript.json count:", len(transcripts))

    if transcripts:
        sample = transcripts[0]
        payload = json.loads(sample.read_text(encoding="utf-8"))
        print("\nSample transcript:", sample)
        print("segments:", len(payload.get("segments", [])))
        print("top-level keys:", sorted(payload.keys()))

windows_dir exists: True
window count: 28

Window: val_001_21001
 - audio.wav: True
 - transcript.json: True
 - summary.json: True

Window: val_002_21002
 - audio.wav: True
 - transcript.json: True
 - summary.json: True

Window: val_003_21003
 - audio.wav: True
 - transcript.json: True
 - summary.json: True

Window: val_004_21005
 - audio.wav: True
 - transcript.json: True
 - summary.json: True

Window: val_005_21006
 - audio.wav: True
 - transcript.json: True
 - summary.json: True

transcript.json count: 28

Sample transcript: /content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows/val_001_21001/transcript.json
segments: 1
top-level keys: ['batch_size', 'batched_mode', 'compute_type', 'device', 'duration_sec', 'language', 'language_probability', 'model_name', 'segments', 'source']


## 12. Упаковка ASR-артефактов

Архив удобно:
- скачать из Colab;
- сохранить на Google Drive;
- использовать как вход для локального diarization / post-processing.

In [ ]:
%cd {DRIVE_DATA_ROOT}
!zip -r "{OUTPUT_DIR}.zip" "{OUTPUT_DIR}"
print("Zip created:", str(OUTPUT_DIR) + ".zip")

/content/drive/MyDrive/AudioIntent/data
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/ (stored 0%)
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/gold.xlsx (deflated 8%)
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows/ (stored 0%)
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows/val_001_21001/ (stored 0%)
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows/val_001_21001/audio.wav (deflated 20%)
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows/val_001_21001/transcript.json (deflated 72%)
updating: content/drive/MyDrive/AudioIntent/data/artifacts/validation_status_svoboden_asr_colab/windows/val_001_21001/summary.json (deflated 42%)
updating: content/drive/MyDrive/AudioIntent/d

In [ ]:
from google.colab import files

zip_path = str(OUTPUT_DIR) + ".zip"
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>